<a href="https://colab.research.google.com/github/leorfoletto/car_prediction/blob/main/bolsa_us.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# =============================================================================
# CÉLULA 1: INSTALAÇÕES E IMPORTS — US MARKET SCREENER
# =============================================================================
# !pip install yfinance pandas plotly lxml html5lib beautifulsoup4 tqdm --quiet --upgrade

import yfinance as yf
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm
import pickle, os, time, random
from datetime import datetime, timedelta
import warnings

warnings.filterwarnings('ignore')
pd.set_option('display.float_format', lambda x: '%.2f' % x)

# --------- CONFIGURAÇÕES PARA MERCADO AMERICANO ---------
GRAHAM_MULTIPLIER      = 22.5     # Fórmula original de Graham
GRAHAM_MARGIN_MIN      = 20       # US exige margem >20% (mercado mais eficiente)
PEG_THRESHOLD          = 1.0      # PEG < 1 = Lynch approved
PEG_GARP_THRESHOLD     = 1.5      # GARP (Growth At Reasonable Price) tolerance

# Setores excluídos da Fórmula Mágica (EV/EBIT não faz sentido para eles)
SETORES_EXCLUIR_MAGICA = ['Financial Services', 'Real Estate', 'Utilities']

# Limites anti-rate-limit
MAX_WORKERS      = 6
REQUEST_DELAY    = 0.15
MAX_RETRIES      = 3
CACHE_DIR        = '/content/cache_us'
CACHE_VALIDADE_H = 12

os.makedirs(CACHE_DIR, exist_ok=True)
print(f"🇺🇸 US Screener configurado. Cache: {CACHE_DIR}")

🇺🇸 US Screener configurado. Cache: /content/cache_us


In [ ]:
# =============================================================================
# CÉLULA 2 (CORRIGIDA): UNIVERSOS COM FALLBACK HARDCODED
# =============================================================================

DOW_30 = [
    'AAPL','AMGN','AXP','BA','CAT','CRM','CSCO','CVX','DIS','DOW',
    'GS','HD','HON','IBM','INTC','JNJ','JPM','KO','MCD','MMM',
    'MRK','MSFT','NKE','PG','TRV','UNH','V','VZ','WMT','WBA'
]

MEGA_CAPS_EXTRAS = [
    'GOOGL','GOOG','META','NVDA','TSLA','BRK-B','LLY','AVGO','ORCL','ADBE',
    'NFLX','AMD','QCOM','TXN','INTU','ISRG','AMAT','BKNG','SYK','PYPL',
    'NOW','UBER','SBUX','GILD','MDLZ','VRTX','ADI','REGN','PANW','CRWD'
]

# S&P 500 completo hardcoded (atualizado 2024-2025)
SP500_HARDCODED = [
    'MMM','AOS','ABT','ABBV','ACN','ADBE','AMD','AES','AFL','A','APD',
    'ABNB','AKAM','ALB','ARE','ALGN','ALLE','LNT','ALL','GOOGL','GOOG',
    'MO','AMZN','AMCR','AEE','AAL','AEP','AXP','AIG','AMT','AWK','AMP',
    'AME','AMGN','APH','ADI','ANSS','AON','APA','AAPL','AMAT','APTV',
    'ACGL','ADM','ANET','AJG','AIZ','T','ATO','ADSK','ADP','AZO','AVB',
    'AVY','AXON','BKR','BALL','BAC','BAX','BDX','BBY','BIO','TECH','BIIB',
    'BLK','BK','BA','BWA','BSX','BMY','AVGO','BR','BRO','BF-B','BLDR',
    'BXP','CHRW','CDNS','CZR','CPT','CPB','COF','CAH','KMX','CCL','CARR',
    'CTLT','CAT','CBOE','CBRE','CDW','CE','COR','CNC','CNP','CF','CHTR',
    'CVX','CMG','CB','CHD','CI','CINF','CTAS','CSCO','C','CFG','CLX',
    'CME','CMS','KO','CTSH','CL','CMCSA','CMA','CAG','COP','ED','STZ',
    'CEG','COO','CPRT','GLW','CPAY','CTVA','CSGP','COST','CTRA','CCI',
    'CSX','CMI','CVS','DHI','DHR','DRI','DVA','DAY','DE','DAL','XRAY',
    'DVN','DXCM','FANG','DLR','DFS','DG','DLTR','D','DPZ','DOV','DOW',
    'DHI','DTE','DUK','DD','EMN','ETN','EBAY','ECL','EIX','EW','EA',
    'ELV','LLY','EMR','ENPH','ETR','EOG','EPAM','EQT','EFX','EQIX','EQR',
    'ESS','EL','ETSY','EG','EVRG','ES','EXC','EXPE','EXPD','EXR','XOM',
    'FFIV','FDS','FICO','FAST','FRT','FDX','FIS','FITB','FSLR','FE',
    'FI','FMC','F','FTNT','FTV','FOXA','FOX','BEN','FCX','GRMN','IT',
    'GE','GEHC','GEV','GEN','GNRC','GD','GIS','GM','GPC','GILD','GPN',
    'GL','GDDY','GS','HAL','HIG','HAS','HCA','DOC','HSIC','HSY','HES',
    'HPE','HLT','HOLX','HD','HON','HRL','HST','HWM','HPQ','HUBB','HUM',
    'HBAN','HII','IBM','IEX','IDXX','ITW','INCY','IR','PODD','INTC',
    'ICE','IFF','IP','IPG','INTU','ISRG','IVZ','INVH','IQV','IRM','JBHT',
    'JBL','JKHY','J','JNJ','JCI','JPM','JNPR','K','KVUE','KDP','KEY',
    'KEYS','KMB','KIM','KMI','KLAC','KHC','KR','LHX','LH','LRCX','LW',
    'LVS','LDOS','LEN','LIN','LYV','LKQ','LMT','L','LOW','LULU','LYB',
    'MTB','MRO','MPC','MKTX','MAR','MMC','MLM','MAS','MA','MTCH','MKC',
    'MCD','MCK','MDT','MRK','META','MET','MTD','MGM','MCHP','MU','MSFT',
    'MAA','MRNA','MHK','MOH','TAP','MDLZ','MPWR','MNST','MCO','MS','MOS',
    'MSI','MSCI','NDAQ','NTAP','NFLX','NEM','NWSA','NWS','NEE','NKE',
    'NI','NDSN','NSC','NTRS','NOC','NCLH','NRG','NUE','NVDA','NVR','NXPI',
    'ORLY','OXY','ODFL','OMC','ON','OKE','ORCL','OTIS','PCAR','PKG','PANW',
    'PARA','PH','PAYX','PAYC','PYPL','PNR','PEP','PFE','PCG','PM','PSX',
    'PNW','PXD','PNC','POOL','PPG','PPL','PFG','PG','PGR','PRU','PEG',
    'PTC','PSA','PHM','QRVO','PWR','QCOM','DGX','RL','RJF','RTX','O',
    'REG','REGN','RF','RSG','RMD','RVTY','ROK','ROL','ROP','ROST','RCL',
    'SPGI','CRM','SBAC','SLB','STX','SRE','NOW','SHW','SPG','SWKS','SJM',
    'SNA','SOLV','SO','LUV','SWK','SBUX','STT','STLD','STE','SYK','SMCI',
    'SYF','SNPS','SYY','TMUS','TROW','TTWO','TPR','TRGP','TGT','TEL',
    'TDY','TFX','TER','TSLA','TXN','TXT','TMO','TJX','TSCO','TT','TDG',
    'TRV','TRMB','TFC','TYL','TSN','USB','UBER','UDR','ULTA','UNP','UAL',
    'UPS','URI','UNH','UHS','VLO','VTR','VLTO','VRSN','VRSK','VZ','VRTX',
    'VTRS','VICI','V','VST','VMC','WRB','GWW','WAB','WBA','WMT','DIS',
    'WBD','WM','WAT','WEC','WFC','WELL','WST','WDC','WY','WHR','WMB',
    'WTW','WYNN','XEL','XYL','YUM','ZBRA','ZBH','ZTS'
]

NASDAQ100_HARDCODED = [
    'ADBE','ADP','ABNB','ALGN','GOOGL','GOOG','AMZN','AMD','AEP','AMGN',
    'ADI','ANSS','AAPL','AMAT','ASML','TEAM','ADSK','AZN','ATVI','AVGO',
    'BIDU','BIIB','BKNG','CDNS','CDW','CHTR','CTAS','CSCO','CTSH','CMCSA',
    'CEG','CPRT','CSGP','COST','CRWD','CSX','DDOG','DXCM','FANG','DLTR',
    'EBAY','EA','EXC','FAST','FTNT','GEHC','GILD','GFS','HON','IDXX',
    'ILMN','INTC','INTU','ISRG','JD','KDP','KLAC','KHC','LRCX','LULU',
    'MELI','MAR','MRVL','MTCH','MU','MSFT','MRNA','MDLZ','MNST','NFLX',
    'NVDA','NXPI','ORLY','ON','PCAR','PANW','PAYX','PYPL','PDD','QCOM',
    'REGN','ROST','CRM','SBUX','SNPS','SIRI','SMCI','TTWO','TMUS','TSLA',
    'TXN','VRSK','VRTX','WBD','WDAY','XEL','ZS','ZM','META','MCHP'
]

def obter_sp500():
    """Tenta scrape; se falhar, usa lista hardcoded."""
    try:
        url = 'https://en.wikipedia.org/wiki/List_of_S%26P_500_companies'
        tabelas = pd.read_html(url, storage_options={"User-Agent": "Mozilla/5.0"})
        sp500 = tabelas[0]['Symbol'].tolist()
        sp500 = [t.replace('.', '-') for t in sp500]
        if len(sp500) > 400:
            print(f"✅ S&P 500 via scrape: {len(sp500)} tickers")
            return sp500
        raise ValueError("Lista incompleta")
    except Exception as e:
        print(f"⚠️  Scrape S&P 500 falhou ({e}). Usando lista hardcoded ({len(SP500_HARDCODED)} tickers).")
        return SP500_HARDCODED


def obter_nasdaq100():
    """Tenta scrape; se falhar, usa lista hardcoded."""
    try:
        url = 'https://en.wikipedia.org/wiki/Nasdaq-100'
        tabelas = pd.read_html(url, storage_options={"User-Agent": "Mozilla/5.0"})
        for t in tabelas:
            cols_str = [str(c).lower() for c in t.columns]
            if any('ticker' in c or 'symbol' in c for c in cols_str):
                col = [c for c in t.columns
                       if 'ticker' in str(c).lower() or 'symbol' in str(c).lower()][0]
                tickers = t[col].astype(str).tolist()
                tickers = [t.replace('.', '-') for t in tickers]
                if len(tickers) > 80:
                    print(f"✅ NASDAQ-100 via scrape: {len(tickers)} tickers")
                    return tickers
        raise ValueError("Coluna não encontrada")
    except Exception as e:
        print(f"⚠️  Scrape NASDAQ-100 falhou ({e}). Usando lista hardcoded ({len(NASDAQ100_HARDCODED)} tickers).")
        return NASDAQ100_HARDCODED


def construir_universo_us(incluir_sp500=True, incluir_nasdaq=False,
                          incluir_dow=False, incluir_mega_extras=False,
                          custom_tickers=None, limite=None):
    universo = []
    if incluir_sp500:       universo += obter_sp500()
    if incluir_nasdaq:      universo += obter_nasdaq100()
    if incluir_dow:         universo += DOW_30
    if incluir_mega_extras: universo += MEGA_CAPS_EXTRAS
    if custom_tickers:      universo += list(custom_tickers)

    # Deduplicação + limpeza
    universo = list(dict.fromkeys(universo))
    universo = [t for t in universo
                if t and isinstance(t, str) and 1 <= len(t) <= 6
                and t not in ('nan','None','')]

    if limite:
        universo = universo[:limite]

    print(f"📊 Universo US construído: {len(universo)} tickers únicos")
    return universo

In [ ]:
# =============================================================================
# CÉLULA 3: EXTRAÇÃO COM CACHE + MÉTRICAS ESPECÍFICAS DE US
# =============================================================================

def _cache_path(ticker: str) -> str:
    return os.path.join(CACHE_DIR, f"{ticker.replace('.','_').replace('-','_')}.pkl")


def _cache_valido(path: str) -> bool:
    if not os.path.exists(path):
        return False
    mtime = datetime.fromtimestamp(os.path.getmtime(path))
    return datetime.now() - mtime < timedelta(hours=CACHE_VALIDADE_H)


def safe_get(info: dict, key: str, default=np.nan):
    val = info.get(key, default) if info else default
    return np.nan if val is None else val


def _extrair_do_yahoo(ticker: str) -> dict:
    """Extrai métricas específicas para análise US, incluindo FCF Yield."""
    tk = yf.Ticker(ticker)
    info = tk.info or {}

    if len(info) < 5:
        raise ValueError(f"Info insuficiente para {ticker}")

    preco = safe_get(info, 'currentPrice') or safe_get(info, 'regularMarketPrice')
    dy    = safe_get(info, 'dividendYield')
    if pd.notna(dy) and dy > 1:
        dy = dy / 100

    # --- EBIT via financials (mais preciso que EBITDA para Greenblatt) ---
    ebit = np.nan
    try:
        fin = tk.financials
        if fin is not None and not fin.empty:
            if 'EBIT' in fin.index:
                ebit = fin.loc['EBIT'].iloc[0]
            elif 'Operating Income' in fin.index:
                ebit = fin.loc['Operating Income'].iloc[0]
    except Exception:
        pass
    if pd.isna(ebit):
        ebit = safe_get(info, 'ebitda')

    # --- ROIC via balance sheet (adaptado para taxonomia US) ---
    roic = np.nan
    try:
        bs = tk.balance_sheet
        if bs is not None and not bs.empty and pd.notna(ebit):
            equity = np.nan
            debt   = np.nan
            # Yahoo usa variações de nomes — tenta todas
            for eq_name in ['Stockholders Equity', 'Total Stockholder Equity', 'Common Stock Equity']:
                if eq_name in bs.index:
                    equity = bs.loc[eq_name].iloc[0]
                    break
            for dt_name in ['Total Debt', 'Long Term Debt', 'Net Debt']:
                if dt_name in bs.index:
                    debt = bs.loc[dt_name].iloc[0]
                    break
            if pd.notna(equity) and pd.notna(debt) and (equity + debt) > 0:
                roic = ebit / (equity + debt)
    except Exception:
        pass
    if pd.isna(roic):
        roic = safe_get(info, 'returnOnAssets')

    # --- Free Cash Flow Yield (métrica chave em análise US) ---
    fcf = safe_get(info, 'freeCashflow')
    market_cap = safe_get(info, 'marketCap')
    fcf_yield = fcf / market_cap if pd.notna(fcf) and pd.notna(market_cap) and market_cap > 0 else np.nan

    # --- 5Y Dividend Growth via histórico (útil para dividend growers) ---
    div_growth_5y = np.nan
    try:
        divs = tk.dividends
        if divs is not None and not divs.empty and len(divs) > 0:
            divs = divs.sort_index()
            hoje = divs.index[-1]
            cinco_anos_atras = hoje - pd.Timedelta(days=5*365)
            divs_recente = divs[divs.index >= hoje - pd.Timedelta(days=365)].sum()
            divs_5y_atras = divs[(divs.index >= cinco_anos_atras) &
                                  (divs.index < cinco_anos_atras + pd.Timedelta(days=365))].sum()
            if divs_5y_atras > 0:
                div_growth_5y = (divs_recente / divs_5y_atras) ** (1/5) - 1
    except Exception:
        pass

    return {
        'Ticker': ticker,
        'Nome': safe_get(info, 'longName', ticker),
        'Setor': safe_get(info, 'sector', 'N/A'),
        'Indústria': safe_get(info, 'industry', 'N/A'),
        'Preço': preco,
        'Market Cap': market_cap,
        'Enterprise Value': safe_get(info, 'enterpriseValue'),
        'Volume Médio': safe_get(info, 'averageVolume'),

        # Múltiplos
        'P/L': safe_get(info, 'trailingPE'),
        'P/L Forward': safe_get(info, 'forwardPE'),
        'P/VP': safe_get(info, 'priceToBook'),
        'P/S': safe_get(info, 'priceToSalesTrailing12Months'),
        'EV/EBITDA': safe_get(info, 'enterpriseToEbitda'),

        # Rentabilidade
        'ROE': safe_get(info, 'returnOnEquity'),
        'ROIC': roic,
        'Margem Operacional': safe_get(info, 'operatingMargins'),
        'Margem Líquida': safe_get(info, 'profitMargins'),

        # Por ação
        'LPA': safe_get(info, 'trailingEps'),
        'LPA Forward': safe_get(info, 'forwardEps'),
        'VPA': safe_get(info, 'bookValue'),

        # Dividendos
        'DY': dy,
        'Div 12M': safe_get(info, 'trailingAnnualDividendRate'),
        'Payout Ratio': safe_get(info, 'payoutRatio'),
        'Div Growth 5Y': div_growth_5y,

        # Crescimento
        'Crescimento Lucros': safe_get(info, 'earningsGrowth'),
        'Crescimento Receita': safe_get(info, 'revenueGrowth'),
        'PEG (yf)': safe_get(info, 'trailingPegRatio') or safe_get(info, 'pegRatio'),

        # Saúde financeira
        'Dívida/Equity': safe_get(info, 'debtToEquity'),
        'Current Ratio': safe_get(info, 'currentRatio'),
        'Beta': safe_get(info, 'beta'),

        # Para Greenblatt
        'EBIT': ebit,
        'FCF': fcf,
        'FCF Yield': fcf_yield,
    }


def extrair_ticker_resiliente(ticker: str, usar_cache: bool = True) -> dict:
    """Cache + retry com backoff exponencial."""
    cache_file = _cache_path(ticker)

    if usar_cache and _cache_valido(cache_file):
        try:
            with open(cache_file, 'rb') as f:
                return pickle.load(f)
        except Exception:
            pass

    ultima_exc = None
    for tentativa in range(MAX_RETRIES):
        try:
            time.sleep(REQUEST_DELAY + random.random() * 0.1)
            dados = _extrair_do_yahoo(ticker)
            try:
                with open(cache_file, 'wb') as f:
                    pickle.dump(dados, f)
            except Exception:
                pass
            return dados
        except Exception as e:
            ultima_exc = e
            time.sleep((2 ** tentativa) + random.random())

    return {'Ticker': ticker, 'erro': str(ultima_exc)}


def extrair_universo(tickers: list, usar_cache: bool = True) -> pd.DataFrame:
    resultados, falhas = [], []

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = {executor.submit(extrair_ticker_resiliente, t, usar_cache): t
                   for t in tickers}
        for fut in tqdm(as_completed(futures), total=len(tickers),
                         desc='📡 Extraindo (US)'):
            res = fut.result()
            (falhas if 'erro' in res else resultados).append(
                res['Ticker'] if 'erro' in res else res
            )

    df = pd.DataFrame([r for r in resultados if isinstance(r, dict)])
    if 'Ticker' in df.columns:
        df = df.set_index('Ticker')

    print(f"\n✅ Sucesso: {len(df)} | ❌ Falhas: {len(falhas)}")
    if falhas and len(falhas) <= 15:
        print(f"   Falhas: {falhas}")
    return df


def limpar_cache():
    import shutil
    if os.path.exists(CACHE_DIR):
        shutil.rmtree(CACHE_DIR); os.makedirs(CACHE_DIR)
    print("🗑️ Cache limpo.")

In [ ]:
# =============================================================================
# CÉLULA 4 (CORRIGIDA): FILTROS COM PROTEÇÃO PARA DATAFRAME VAZIO
# =============================================================================

def filtrar_qualidade_us(df: pd.DataFrame,
                          market_cap_min: float = 2e9,
                          volume_min: float = 5e5,
                          exigir_fundamentos: bool = True,
                          min_campos_validos: int = 10) -> pd.DataFrame:

    # ── Guard: DataFrame vazio ──────────────────────────────────────────────
    if df.empty:
        print("❌ DataFrame vazio — nenhum ticker foi extraído com sucesso.")
        print("   Verifique sua conexão ou reduza MAX_WORKERS para evitar rate limit.")
        return df

    n_antes = len(df)
    df = df.copy()

    # Garante que as colunas esperadas existam antes de filtrar
    if 'Market Cap' in df.columns and market_cap_min:
        df = df[df['Market Cap'].fillna(0) >= market_cap_min]

    if 'Volume Médio' in df.columns and volume_min:
        df = df[df['Volume Médio'].fillna(0) >= volume_min]

    if exigir_fundamentos:
        cols_fund = [c for c in ['LPA','VPA'] if c in df.columns]
        if cols_fund:
            df = df.dropna(subset=cols_fund, how='any')

    # Campos críticos — só usa os que realmente existem no DataFrame
    cols_criticas_candidatas = ['Preço','P/L','P/VP','ROE','LPA','VPA',
                                 'Market Cap','Enterprise Value',
                                 'Margem Operacional','EBIT']
    cols_criticas = [c for c in cols_criticas_candidatas if c in df.columns]

    if cols_criticas and min_campos_validos:
        df['_validos'] = df[cols_criticas].notna().sum(axis=1)
        df = df[df['_validos'] >= min(min_campos_validos, len(cols_criticas))]
        df = df.drop(columns='_validos')

    n_depois = len(df)
    print(f"🧹 Filtro de qualidade: {n_antes} → {n_depois} "
          f"({n_antes - n_depois} removidas)")
    return df


def diagnostico_cobertura(df: pd.DataFrame):
    if df.empty:
        print("⚠️ DataFrame vazio — sem dados para diagnóstico.")
        return pd.DataFrame()
    cobertura = (df.notna().sum() / len(df) * 100).sort_values(ascending=False)
    print("\n📊 Cobertura de dados por campo (%):")
    return pd.DataFrame({'% Preenchimento': cobertura}).round(1)

In [ ]:
# =============================================================================
# CÉLULA 5: 3 ESTRATÉGIAS PARA US (Graham, Lynch, Fórmula Mágica)
# =============================================================================

def aplicar_graham_us(df):
    """
    Graham para US com critério mais rigoroso: exige margem > 20%
    (mercado mais eficiente que o BR, poucas oportunidades óbvias).
    """
    mask = (df['LPA'] > 0) & (df['VPA'] > 0)
    df['VI Graham']            = np.where(mask,
                                  np.sqrt(GRAHAM_MULTIPLIER * df['VPA'] * df['LPA']),
                                  np.nan)
    df['Margem Segurança (%)'] = (df['VI Graham'] - df['Preço']) / df['Preço'] * 100

    # Critério US: margem > 20% (mais rigoroso que BR)
    df['✓ Graham'] = df['Margem Segurança (%)'] > GRAHAM_MARGIN_MIN
    return df


def aplicar_lynch_us(df):
    """
    Peter Lynch adaptado: prioriza PEG do yfinance (forward-looking),
    com fallback para cálculo manual.
    Adiciona categoria GARP (Growth At Reasonable Price) para PEG entre 1 e 1.5.
    """
    cresc_pct = df['Crescimento Lucros'] * 100
    peg_calc = np.where(
        (df['P/L'] > 0) & (cresc_pct > 0),
        df['P/L'] / cresc_pct,
        np.nan
    )
    df['PEG'] = df['PEG (yf)'].fillna(pd.Series(peg_calc, index=df.index))

    df['✓ Lynch']      = (df['PEG'] > 0) & (df['PEG'] < PEG_THRESHOLD)
    df['✓ Lynch GARP'] = (df['PEG'] > 0) & (df['PEG'] < PEG_GARP_THRESHOLD)
    return df


def aplicar_formula_magica_us(df, top_n_pct=0.25, excluir_setores=True):
    """
    Greenblatt adaptado para US:
      • Exclui bancos, REITs, utilities (EV/EBIT perde sentido)
      • Top 25% vencem (mais rigoroso que BR por ter mais ações)
    """
    df['Earnings Yield'] = np.where(
        (df['Enterprise Value'] > 0) & df['EBIT'].notna(),
        df['EBIT'] / df['Enterprise Value'],
        np.nan
    )

    if excluir_setores:
        mask_setor = ~df['Setor'].isin(SETORES_EXCLUIR_MAGICA)
    else:
        mask_setor = pd.Series(True, index=df.index)

    mask_valido = df['Earnings Yield'].notna() & df['ROIC'].notna() & mask_setor

    # Rankings só aplicados a empresas elegíveis
    df['Rank EY']   = np.nan
    df['Rank ROIC'] = np.nan
    df.loc[mask_valido, 'Rank EY']   = df.loc[mask_valido, 'Earnings Yield'].rank(ascending=False, method='min')
    df.loc[mask_valido, 'Rank ROIC'] = df.loc[mask_valido, 'ROIC'].rank(ascending=False, method='min')
    df['Rank Mágica Total'] = df['Rank EY'] + df['Rank ROIC']

    threshold = df['Rank Mágica Total'].quantile(top_n_pct)
    df['✓ Fórmula Mágica'] = df['Rank Mágica Total'] <= threshold
    return df


def score_final_us(df):
    """
    Score final adaptado para 3 estratégias (0-3) e score contínuo ponderado
    com métricas típicas do mercado US.
    """
    cols = ['✓ Graham', '✓ Lynch', '✓ Fórmula Mágica']
    df['Score (0-3)'] = df[cols].fillna(False).sum(axis=1)

    # Score contínuo: pondera múltiplas métricas em percentis
    metricas_peso = {
        'Margem Segurança (%)': 0.20,   # Valor (Graham)
        'ROIC':                 0.20,   # Qualidade
        'Earnings Yield':       0.15,   # Valor (Greenblatt)
        'FCF Yield':            0.15,   # Geração de caixa (crucial em US)
        'ROE':                  0.10,   # Rentabilidade
        'Margem Operacional':   0.10,   # Qualidade operacional
        'Crescimento Lucros':   0.10,   # Crescimento
    }

    score_continuo = pd.Series(0.0, index=df.index)
    for col, peso in metricas_peso.items():
        if col in df.columns:
            pct = df[col].rank(pct=True)
            score_continuo += pct.fillna(0) * peso

    df['Score Contínuo'] = (score_continuo * 100).round(2)
    return df


def rodar_screening_us(tickers: list, filtros: dict = None) -> pd.DataFrame:
    """Pipeline completo de screening US."""
    filtros = filtros or {}

    print(f"\n🔄 Etapa 1/3: Extraindo {len(tickers)} tickers...")
    df = extrair_universo(tickers)

    print(f"\n🔄 Etapa 2/3: Filtros de qualidade...")
    df = filtrar_qualidade_us(df, **filtros)

    print(f"\n🔄 Etapa 3/3: Aplicando estratégias...")
    df = aplicar_graham_us(df)
    df = aplicar_lynch_us(df)
    df = aplicar_formula_magica_us(df)
    df = score_final_us(df)

    print(f"\n✅ {len(df)} ações no ranking final.")
    return df

In [ ]:
# =============================================================================
# CÉLULA 6: STYLING PARA TABELAS
# =============================================================================

def destacar_bool(val):
    if pd.isna(val):  return 'background-color: #f0f0f0; color: #888'
    if val is True:   return 'background-color: #90EE90; color: #003300; font-weight: bold'
    if val is False:  return 'background-color: #FFB6B6; color: #660000'
    return ''


def destacar_score_3(val):
    """Gradiente para score de 0-3 (em vez de 0-4)."""
    if pd.isna(val): return ''
    cores = {0:'#FFB6B6', 1:'#FFE699', 2:'#B3E6B3', 3:'#66CC66'}
    return f'background-color: {cores.get(int(val), "#FFFFFF")}; font-weight: bold'


def formatar_tabela_us(df, top_n: int = 30):
    """Exibe Top N com todas as métricas relevantes para US."""
    cols_bool = ['✓ Graham','✓ Lynch','✓ Fórmula Mágica']
    cols_show = ['Nome','Setor','Preço','Market Cap',
                 'P/L','P/L Forward','P/VP','EV/EBITDA',
                 'ROE','ROIC','Margem Operacional',
                 'FCF Yield','DY',
                 'Margem Segurança (%)','✓ Graham',
                 'PEG','Crescimento Lucros','✓ Lynch',
                 'Earnings Yield','✓ Fórmula Mágica',
                 'Score (0-3)','Score Contínuo']
    cols_show = [c for c in cols_show if c in df.columns]

    df_top = (df.sort_values(['Score (0-3)','Score Contínuo'], ascending=False)
                .head(top_n)[cols_show])

    styled = (df_top.style
              .applymap(destacar_bool, subset=[c for c in cols_bool if c in df_top.columns])
              .applymap(destacar_score_3, subset=['Score (0-3)'])
              .background_gradient(subset=['Score Contínuo'], cmap='RdYlGn')
              .format({
                  'Preço':'${:.2f}',
                  'Market Cap':'${:,.0f}',
                  'P/L':'{:.2f}','P/L Forward':'{:.2f}','P/VP':'{:.2f}','EV/EBITDA':'{:.2f}',
                  'ROE':'{:.2%}','ROIC':'{:.2%}','Margem Operacional':'{:.2%}',
                  'FCF Yield':'{:.2%}','DY':'{:.2%}',
                  'Earnings Yield':'{:.2%}','Crescimento Lucros':'{:.1%}',
                  'Margem Segurança (%)':'{:.1f}%',
                  'PEG':'{:.2f}',
                  'Score (0-3)':'{:.0f}','Score Contínuo':'{:.1f}',
              }, na_rep='—')
              .set_caption(f'🇺🇸 Top {top_n} — Screening Multi-Estratégia US'))
    return styled


def filtrar_por_estrategia(df, estrategia: str):
    col = f'✓ {estrategia}'
    return df[df[col] == True].sort_values('Score Contínuo', ascending=False)


def resumo_por_setor_us(df):
    """Panorama setorial."""
    resumo = (df.groupby('Setor')
                .agg(N_Empresas=('Preço','count'),
                     Score_Medio=('Score (0-3)','mean'),
                     ROIC_Mediano=('ROIC','median'),
                     PL_Mediano=('P/L','median'),
                     FCF_Yield_Mediano=('FCF Yield','median'),
                     Margem_Op_Mediana=('Margem Operacional','median'))
                .sort_values('Score_Medio', ascending=False))
    return resumo.style.background_gradient(cmap='RdYlGn', subset=['Score_Medio'])

In [ ]:
# =============================================================================
# CÉLULA 7: GRÁFICOS PLOTLY
# =============================================================================

def plot_magica_vs_fcf(df, destacar_top=20):
    """
    Adaptado para US: ROIC × Earnings Yield com FCF Yield como tamanho
    (FCF é mais informativo que DY no mercado US, onde muita empresa
    não paga dividendo mas gera caixa forte).
    """
    df_plot = df.reset_index().dropna(subset=['ROIC','Earnings Yield']).copy()
    df_plot['FCF_size'] = df_plot['FCF Yield'].fillna(0).clip(lower=0.001, upper=0.20) * 100

    df_plot['ROIC_plot'] = df_plot['ROIC'].clip(lower=-0.3, upper=0.8)
    df_plot['EY_plot']   = df_plot['Earnings Yield'].clip(lower=-0.1, upper=0.3)

    fig = px.scatter(
        df_plot,
        x='ROIC_plot', y='EY_plot',
        size='FCF_size',
        color='Score (0-3)',
        color_continuous_scale='RdYlGn',
        hover_name='Ticker',
        hover_data={'Nome':True,'Setor':True,'Preço':':.2f',
                    'ROIC':':.2%','Earnings Yield':':.2%','FCF Yield':':.2%',
                    'Score (0-3)':True,'ROIC_plot':False,'EY_plot':False,'FCF_size':False},
        size_max=40,
        title=f'<b>🇺🇸 Fórmula Mágica × FCF Yield — {len(df_plot)} ações</b><br>'
              f'<sup>Top-direita = zona Greenblatt. Bolhas grandes = forte geração de FCF.</sup>',
    )

    top = df_plot.nlargest(destacar_top, 'Score Contínuo')
    for _, row in top.iterrows():
        fig.add_annotation(x=row['ROIC_plot'], y=row['EY_plot'],
                           text=row['Ticker'], showarrow=False,
                           font=dict(size=9), yshift=10)

    fig.add_hline(y=df_plot['EY_plot'].median(), line_dash='dash',
                  line_color='gray', opacity=0.4)
    fig.add_vline(x=df_plot['ROIC_plot'].median(), line_dash='dash',
                  line_color='gray', opacity=0.4)

    fig.update_layout(
        xaxis=dict(tickformat='.0%', title='ROIC'),
        yaxis=dict(tickformat='.0%', title='Earnings Yield'),
        template='plotly_white', height=700,
    )
    fig.show()


def plot_growth_vs_value(df, destacar_top=15):
    """
    Quadrante clássico US: Crescimento × Valor (P/L).
    Canto sup-esquerdo = GARP (crescimento a preço razoável) = Peter Lynch.
    """
    df_plot = df.reset_index().dropna(subset=['P/L','Crescimento Lucros']).copy()
    df_plot = df_plot[(df_plot['P/L'] > 0) & (df_plot['P/L'] < 100)]   # remove outliers
    df_plot = df_plot[df_plot['Crescimento Lucros'].between(-0.5, 1.5)]

    fig = px.scatter(
        df_plot,
        x='P/L', y='Crescimento Lucros',
        size='Market Cap',
        color='Score (0-3)',
        color_continuous_scale='RdYlGn',
        hover_name='Ticker',
        hover_data={'Nome':True,'Setor':True,'PEG':':.2f','ROIC':':.2%'},
        size_max=50,
        title='<b>🇺🇸 Crescimento vs. Valor — Quadrante Lynch (GARP)</b><br>'
              '<sup>Canto superior esquerdo = alto crescimento a baixo P/L (PEG<1).</sup>',
    )

    # Linha de referência: PEG = 1 (Y = X/100)
    pls = np.linspace(1, df_plot['P/L'].max(), 100)
    fig.add_trace(go.Scatter(
        x=pls, y=pls/100,   # quando PEG=1: growth (decimal) = P/L / 100
        mode='lines', line=dict(dash='dash', color='blue', width=1),
        name='PEG = 1 (Lynch fair)', showlegend=True
    ))

    top = df_plot.nlargest(destacar_top, 'Score Contínuo')
    for _, row in top.iterrows():
        fig.add_annotation(x=row['P/L'], y=row['Crescimento Lucros'],
                           text=row['Ticker'], showarrow=False,
                           font=dict(size=9), yshift=10)

    fig.update_layout(
        xaxis=dict(title='P/L (trailing)'),
        yaxis=dict(tickformat='.0%', title='Crescimento de Lucros (YoY)'),
        template='plotly_white', height=650,
    )
    fig.show()


def plot_heatmap_setorial_us(df):
    """Heatmap: aprovações por setor × estratégia."""
    cols = ['✓ Graham','✓ Lynch','✓ Fórmula Mágica']
    heat = df.groupby('Setor')[cols].sum().astype(int)
    heat.columns = [c.replace('✓ ','') for c in heat.columns]

    fig = px.imshow(heat, text_auto=True, aspect='auto',
                    color_continuous_scale='Greens',
                    title='<b>🇺🇸 Aprovações por Setor × Estratégia</b>',
                    labels=dict(x='Estratégia', y='Setor', color='Nº'))
    fig.update_layout(height=max(400, len(heat)*30), template='plotly_white')
    fig.show()


def plot_distribuicao_scores_us(df):
    fig = px.histogram(
        df.reset_index(), x='Score Contínuo', color='Score (0-3)',
        nbins=30, color_discrete_sequence=px.colors.sequential.RdYlGn,
        title='<b>🇺🇸 Distribuição do Score Contínuo</b>',
    )
    fig.update_layout(template='plotly_white', height=450, bargap=0.05)
    fig.show()


def plot_top_por_estrategia_us(df, top_n=15):
    """Top 15 em cada uma das 3 estratégias (subplots 1x3)."""
    from plotly.subplots import make_subplots

    specs = [
        ('Graham',         'Margem Segurança (%)', False),
        ('Lynch',          'PEG',                  True),
        ('Fórmula Mágica', 'Rank Mágica Total',    True),
    ]

    fig = make_subplots(rows=1, cols=3,
                        subplot_titles=[f'Top {top_n} — {s[0]}' for s in specs],
                        horizontal_spacing=0.12)

    for i, (nome, col, asc) in enumerate(specs):
        aprovados = df[df[f'✓ {nome}'] == True].copy()
        if aprovados.empty: continue
        top = aprovados.nsmallest(top_n, col) if asc else aprovados.nlargest(top_n, col)

        fig.add_trace(
            go.Bar(x=top[col], y=top.index, orientation='h',
                   marker_color='#2E86C1', showlegend=False,
                   text=top[col].round(2), textposition='outside'),
            row=1, col=i+1
        )

    fig.update_layout(height=550, template='plotly_white',
                      title_text=f'<b>🇺🇸 Top {top_n} por Estratégia</b>')
    fig.show()


def plot_dividend_growers(df, top_n=20):
    """
    Bônus US-específico: melhores "dividend growers" — empresas que
    combinam DY razoável + alto crescimento dos dividendos nos últimos 5 anos.
    Estratégia popular no mercado americano.
    """
    df_div = df.reset_index().dropna(subset=['DY','Div Growth 5Y']).copy()
    df_div = df_div[(df_div['DY'] > 0.005) & (df_div['DY'] < 0.10)]
    df_div = df_div[df_div['Div Growth 5Y'].between(-0.1, 0.5)]

    if df_div.empty:
        print("⚠️ Dados de crescimento de dividendos insuficientes.")
        return

    # Score: DY + Div Growth ponderados (Chowder Rule approx)
    df_div['Chowder'] = df_div['DY'] * 100 + df_div['Div Growth 5Y'] * 100
    top = df_div.nlargest(top_n, 'Chowder')

    fig = px.bar(top, x='Chowder', y='Ticker', orientation='h',
                 color='DY', color_continuous_scale='Blues',
                 hover_data={'Nome':True,'DY':':.2%','Div Growth 5Y':':.2%'},
                 title=f'<b>🇺🇸 Top {top_n} Dividend Growers (Chowder Rule: DY + Div Growth)</b>',
                 text='Chowder')
    fig.update_traces(texttemplate='%{text:.1f}', textposition='outside')
    fig.update_layout(template='plotly_white', height=600,
                      yaxis=dict(categoryorder='total ascending'))
    fig.show()

In [ ]:
# =============================================================================
# CÉLULA 8 (CORRIGIDA): EXECUÇÃO COM VALIDAÇÕES EM CADA ETAPA
# =============================================================================

# ── 1. Montar universo ──────────────────────────────────────────────────────
universo = construir_universo_us(
    incluir_sp500       = True,
    incluir_nasdaq      = False,
    incluir_dow         = False,
    incluir_mega_extras = False,
    # limite = 50,       # ← descomente para testar rápido primeiro
)

# Validação: para tudo se o universo estiver vazio
assert len(universo) > 0, "❌ Universo vazio. Verifique as funções de construção."

# ── 2. Filtros ──────────────────────────────────────────────────────────────
filtros = dict(
    market_cap_min     = 2e9,
    volume_min         = 5e5,
    exigir_fundamentos = True,
    min_campos_validos = 8,    # reduzido de 10 → 8 para maior cobertura
)

# ── 3. Rodar pipeline ───────────────────────────────────────────────────────
df_us = rodar_screening_us(universo, filtros=filtros)

# ── 4. Diagnóstico ──────────────────────────────────────────────────────────
if not df_us.empty:
    print(f"\n📈 {len(df_us)} ações passaram pelos filtros de qualidade.")
    diagnostico_cobertura(df_us)
else:
    print("\n❌ Nenhuma ação sobrou após os filtros.")
    print("💡 Tente reduzir 'market_cap_min' para 5e8 ou 'min_campos_validos' para 6.")

✅ S&P 500 via scrape: 503 tickers
📊 Universo US construído: 503 tickers únicos

🔄 Etapa 1/3: Extraindo 503 tickers...


📡 Extraindo (US):   0%|          | 0/503 [00:00<?, ?it/s]


✅ Sucesso: 503 | ❌ Falhas: 0

🔄 Etapa 2/3: Filtros de qualidade...
🧹 Filtro de qualidade: 503 → 487 (16 removidas)

🔄 Etapa 3/3: Aplicando estratégias...

✅ 487 ações no ranking final.

📈 487 ações passaram pelos filtros de qualidade.

📊 Cobertura de dados por campo (%):


In [ ]:
# Célula 8b — Top 30 (só roda se df não estiver vazio)
if not df_us.empty:
    formatar_tabela_us(df_us, top_n=30)
else:
    print("⚠️ Sem dados para exibir.")

In [ ]:
# Célula 8c — Top Graham
if not df_us.empty and '✓ Graham' in df_us.columns:
    resultado = filtrar_por_estrategia(df_us, 'Graham')
    if not resultado.empty:
        print(f"🎯 Top 20 Graham (Deep Value) — {len(resultado)} aprovadas:")
        display(resultado.head(20)[
            ['Nome','Setor','Preço','P/L','P/VP','Margem Segurança (%)','Score (0-3)']
        ])
    else:
        print("ℹ️ Nenhuma ação passou no critério Graham com os filtros atuais.")

🎯 Top 20 Graham (Deep Value) — 18 aprovadas:


,Nome,Setor,Preço,P/L,P/VP,Margem Segurança (%),Score (0-3)
Ticker,,,,,,,
ALL,The Allstate Corporation,Financial Services,217.18,5.71,1.97,41.34,2
UHS,"Universal Health Services, Inc.",Healthcare,181.69,7.87,1.52,36.97,2
CHTR,"Charter Communications, Inc.",Communication Services,243.06,6.71,1.92,32.21,2
EIX,Edison International,Utilities,70.35,6.09,1.58,52.68,1
ACGL,Arch Capital Group Ltd.,Financial Services,98.25,8.47,1.50,32.95,2
T,AT&T Inc.,Communication Services,26.62,8.76,1.70,23.10,2
CMCSA,Comcast Corporation,Communication Services,31.95,5.93,1.19,78.69,2
VICI,VICI Properties Inc.,Real Estate,28.39,10.88,1.09,37.68,1
AIG,"American International Group, Inc.",Financial Services,76.19,14.03,1.00,26.85,1


Error: Runtime no longer has a reference to this dataframe, please re-run this cell and try again.


In [ ]:
# Célula 8d — Top Lynch
if not df_us.empty and '✓ Lynch' in df_us.columns:
    resultado = filtrar_por_estrategia(df_us, 'Lynch')
    if not resultado.empty:
        print(f"🚀 Top 20 Lynch (GARP) — {len(resultado)} aprovadas:")
        display(resultado.head(20)[
            ['Nome','Setor','P/L','PEG','Crescimento Lucros','ROIC','Score (0-3)']
        ])
    else:
        print("ℹ️ Nenhuma ação passou no critério Lynch com os filtros atuais.")

🚀 Top 20 Lynch (GARP) — 104 aprovadas:


,Nome,Setor,P/L,PEG,Crescimento Lucros,ROIC,Score (0-3)
Ticker,,,,,,,
ALL,The Allstate Corporation,Financial Services,5.71,0.47,1.03,0.36,2
CINF,Cincinnati Financial Corporation,Financial Services,11.08,0.16,0.67,0.18,1
HIG,"The Hartford Insurance Group, Inc.",Financial Services,10.51,0.28,0.38,0.21,1
PYPL,"PayPal Holdings, Inc.",Financial Services,9.22,0.93,0.39,0.22,1
ADBE,Adobe Inc.,Technology,13.95,0.73,0.11,0.49,2
CTSH,Cognizant Technology Solutions Corporation,Technology,12.11,0.96,0.22,0.22,2
INCY,Incyte Corporation,Healthcare,14.93,0.35,0.44,0.32,2
LULU,lululemon athletica inc.,Consumer Cyclical,10.89,0.78,-0.18,0.33,2
CHTR,"Charter Communications, Inc.",Communication Services,6.71,0.40,0.02,0.11,2


Error: Runtime no longer has a reference to this dataframe, please re-run this cell and try again.


In [ ]:
# Célula 8e — Top Fórmula Mágica
if not df_us.empty and '✓ Fórmula Mágica' in df_us.columns:
    resultado = filtrar_por_estrategia(df_us, 'Fórmula Mágica')
    if not resultado.empty:
        print(f"💎 Top 20 Fórmula Mágica — {len(resultado)} aprovadas:")
        display(resultado.head(20)[
            ['Nome','Setor','ROIC','Earnings Yield','FCF Yield','Rank Mágica Total','Score (0-3)']
        ])
    else:
        print("ℹ️ Nenhuma ação passou na Fórmula Mágica com os filtros atuais.")

💎 Top 20 Fórmula Mágica — 90 aprovadas:


,Nome,Setor,ROIC,Earnings Yield,FCF Yield,Rank Mágica Total,Score (0-3)
Ticker,,,,,,,
APA,APA Corporation,Energy,0.29,0.16,0.14,60.00,1
CF,"CF Industries Holdings, Inc.",Basic Materials,0.28,0.10,0.07,84.00,1
ADBE,Adobe Inc.,Technology,0.49,0.09,0.10,47.00,2
SOLV,Solventum Corporation,Healthcare,0.20,0.13,0.05,112.00,1
NEM,Newmont Corporation,Basic Materials,0.29,0.10,0.08,79.00,1
CTSH,Cognizant Technology Solutions Corporation,Technology,0.22,0.13,0.07,95.00,2
DECK,Deckers Outdoor Corporation,Consumer Cyclical,0.45,0.09,0.05,43.00,1
INCY,Incyte Corporation,Healthcare,0.32,0.11,0.03,59.00,2
LULU,lululemon athletica inc.,Consumer Cyclical,0.33,0.12,0.05,48.00,2


In [ ]:
# Célula 8f — Panorama setorial
if not df_us.empty:
    display(resumo_por_setor_us(df_us))

,N_Empresas,Score_Medio,ROIC_Mediano,PL_Mediano,FCF_Yield_Mediano,Margem_Op_Mediana
Setor,,,,,,
Technology,81,0.592593,0.159505,32.643857,0.032150,0.247690
Healthcare,57,0.543860,0.133781,25.270158,0.044337,0.196700
Consumer Cyclical,52,0.538462,0.171228,25.587412,0.031328,0.122515
Communication Services,24,0.500000,0.118293,29.060834,0.054091,0.162115
Consumer Defensive,35,0.457143,0.138696,23.476676,0.049356,0.142590
Financial Services,67,0.432836,0.106965,16.453626,0.060117,0.322170
Energy,22,0.409091,0.126410,22.295415,0.033851,0.166200
Basic Materials,20,0.350000,0.108141,32.767056,0.020231,0.142795
Industrials,67,0.298507,0.146564,30.993507,0.030267,0.163650


In [ ]:
# Célula 8g — Gráficos (só plota se tiver dados suficientes)
if not df_us.empty:
    plot_magica_vs_fcf(df_us, destacar_top=15)
    plot_growth_vs_value(df_us, destacar_top=15)
    plot_distribuicao_scores_us(df_us)
    plot_heatmap_setorial_us(df_us)
    plot_top_por_estrategia_us(df_us, top_n=15)
    plot_dividend_growers(df_us, top_n=20)

AttributeError: module '_plotly_utils.colors.sequential' has no attribute 'RdYlGn'

In [ ]:
# Célula 8h — Exportar Excel
if not df_us.empty:
    from google.colab import files
    caminho = '/content/screening_us.xlsx'
    df_us.to_excel(caminho, index=True)
    files.download(caminho)
    print(f"✅ Arquivo exportado: {caminho}")